Precision, Recall, F1-score
Clase 1 = impago

Precision: De los clientes que el modelo predice como impago, cuántos realmente incumplen.

Recall: De todos los clientes que realmente incumplen, cuántos el modelo predice correctamente.

F1-score: Media armónica entre precision y recall, útil para balancear ambos.


Tu objetivo es detectar impagos (clase 1). Por tanto:

Recall de clase 1 es la métrica más importante
→ quieres minimizar falsos negativos (clientes que incumplen pero tu modelo dice que no).

Precision importa menos que recall si estás dispuesto a aceptar algunos falsos positivos (alertas de riesgo innecesarias).

F1-score de clase 1 te da un balance, útil para comparar modelos.

ROC-AUC es buena métrica general de ranking de riesgo.

In [20]:
import numpy as np
import pandas as pd
import os
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    recall_score,
    precision_score,   
    make_scorer,
    silhouette_score,
    precision_recall_curve,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_samples
)

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestCentroid

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier,
    AdaBoostClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import time

In [2]:
# 1. CARGAR DATOS
def cargar_y_preparar_datos(ruta_archivo):
    df = pd.read_excel(ruta_archivo)
    # Filtrar solo vivienda y copiar para evitar warnings
    df_viv = df[df['Proposito'].astype(str)
                .str.contains('Vivienda', case=False, na=False)].copy()
    # Label
    df_viv['Impago_Label'] = df_viv['Impago'].map({0:0, 1:1})
    return df_viv

# Ajusta esta ruta si es necesario
ruta_real = os.path.join('..', 'Datos', 'Limpios', 'información_préstamos_limpio.xlsx')

if os.path.exists(ruta_real):
    df = cargar_y_preparar_datos(ruta_real)
else:
    print(f" ATENCIÓN: No se encuentra el archivo en {ruta_real}")
    df = pd.DataFrame() 

In [3]:
# 2. DEFINIR X e y
if not df.empty:
    target_col = "Impago_Label"
    columnas_a_eliminar = ["ID", "Impago", "Prima", "Proposito"]

    y = df[target_col]
    X = df.drop(columns=[target_col])
    X = X.drop(columns=[col for col in columnas_a_eliminar if col in X.columns])

    # Eliminar alta cardinalidad
    high_card_cols = [col for col in X.columns if X[col].nunique() > 50]
    X = X.drop(columns=high_card_cols)

    # One-hot encoding
    cat_cols = X.select_dtypes(include="object").columns
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

    X = X.astype("float32")
    
# 3. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

In [ ]:
# 4. CLUSTERING: TORNEO K-MEANS vs AGLOMERATIVO
print("--- Iniciando Optimización de Clustering ---")

#1. Escalado
scaler_cluster = StandardScaler()
X_train_cluster = scaler_cluster.fit_transform(X_train)
X_test_cluster = scaler_cluster.transform(X_test)

# ENCONTRAR EL NÚMERO DE CLUSTERS (K) ÓPTIMO (Probamos de 2 a 5 clusters y nos quedamos con el mejor)

print(" Buscando el número óptimo de clusters (k)...")
best_k = 3  
best_k_score = -1

for k in [2, 3, 4, 5]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_train_cluster)
    score = silhouette_score(X_train_cluster, labels)
    print(f"   k={k} -> Silhouette: {score:.4f}")
        
    if score > best_k_score:
        best_k_score = score
        best_k = k

print(f" Número óptimo seleccionado: k={best_k}")

#PASO 2: TORNEO CON EL K GANADOR (KMeans vs Aglomerativo)
print(f"\n--- Iniciando Torneo (usando k={best_k}) ---")

scores = {}
labels_storage = {}

# Función auxiliar
def asignar_clusters(model, X_train_scaled, X_test_scaled):
    labels_train = model.fit_predict(X_train_scaled)
    if hasattr(model, "predict"):
        labels_test = model.predict(X_test_scaled)
    else:
        centroid_clf = NearestCentroid()
        centroid_clf.fit(X_train_scaled, labels_train)
        labels_test = centroid_clf.predict(X_test_scaled)
        
    return labels_train, labels_test  
# Opción A: KMeans
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
k_labels_train, k_labels_test = asignar_clusters(kmeans, X_train_cluster, X_test_cluster)
scores["KMeans"] = silhouette_score(X_train_cluster, k_labels_train)
labels_storage["KMeans"] = (k_labels_train, k_labels_test)
print(f"Silhouette KMeans: {scores['KMeans']:.4f}")

# Opción B: Aglomerativo
agg = AgglomerativeClustering(n_clusters=best_k)
a_labels_train, a_labels_test = asignar_clusters(agg, X_train_cluster, X_test_cluster)
scores["Agglomerative"] = silhouette_score(X_train_cluster, a_labels_train)
labels_storage["Agglomerative"] = (a_labels_train, a_labels_test)
print(f"Silhouette Agglomerative: {scores['Agglomerative']:.4f}")

#SELECCIÓN FINAL
best_model_name = max(scores, key=scores.get)
print(f"GANADOR DEL TORNEO: {best_model_name} con k={best_k}")

# Aplicar al dataset
final_labels_train, final_labels_test = labels_storage[best_model_name]

# One-Hot Encoding
train_dummies = pd.get_dummies(final_labels_train, prefix='Cluster_Group')
test_dummies = pd.get_dummies(final_labels_test, prefix='Cluster_Group')

# Alinear columnas
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

# Unir
train_dummies.index = X_train.index
test_dummies.index = X_test.index
X_train = pd.concat([X_train, train_dummies], axis=1)
X_test = pd.concat([X_test, test_dummies], axis=1)

print(f" Variables de cluster añadidas. Nuevas columnas: {list(train_dummies.columns)}")


--- Iniciando Optimización de Clustering ---
 Buscando el número óptimo de clusters (k)...


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\alari\.conda\envs\RETO_07_MORADO\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\alari\.conda\envs\RETO_07_MORADO\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\alari\.conda\envs\RETO_07_MORADO\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\alari\.conda\envs\RETO_07_MORADO\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa2 in position 116: invalid start byte


   k=2 -> Silhouette: 0.1469
   k=3 -> Silhouette: 0.1329
   k=4 -> Silhouette: 0.1505
   k=5 -> Silhouette: 0.1456
 Número óptimo seleccionado: k=4

--- Iniciando Torneo (usando k=4) ---
Silhouette KMeans: 0.1505
Silhouette Agglomerative: 0.1801
GANADOR DEL TORNEO: Agglomerative con k=4
 Variables de cluster añadidas. Nuevas columnas: ['Cluster_Group_0', 'Cluster_Group_1', 'Cluster_Group_2', 'Cluster_Group_3']


In [ ]:
# 4. CLUSTERING AVANZADO: TORNEO, MÉTRICAS E INYECCIÓN
print("--- Iniciando Optimización Avanzada de Clustering ---")

#Escalado de los datos (Vital para Clustering)
scaler_cluster = StandardScaler()
X_train_cluster = scaler_cluster.fit_transform(X_train)
X_test_cluster = scaler_cluster.transform(X_test)

#PRUEBA DE DBSCAN (Descarte por densidad)
print("Probando DBSCAN (Basado en Densidad)...")
dbscan = DBSCAN(eps=2.0, min_samples=10)
db_labels = dbscan.fit_predict(X_train_cluster)
n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_ruido = list(db_labels).count(-1)
print(f"DBSCAN encontró {n_clusters_db} clusters y {n_ruido} puntos de ruido (outliers).")
print("Motivo de descarte: En datos financieros continuos, DBSCAN suele agrupar casi todo en un solo clúster gigante o generar demasiado ruido. Pasamos a algoritmos particionales.\n")

#TORNEO K-MEANS vs AGLOMERATIVO (Guardando todas las métricas)
print("KMeans vs Jerárquico/Aglomerativo...")
resultados_clustering = []
k_values = [2, 3, 4, 5]

for k in k_values:
    # --- Modelo K-Means ---
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_labels = km.fit_predict(X_train_cluster)
    
    resultados_clustering.append({
        'Modelo': 'K-Means',
        'K': k,
        'Silhouette (↑)': silhouette_score(X_train_cluster, km_labels),
        'Calinski-Harabasz (↑)': calinski_harabasz_score(X_train_cluster, km_labels),
        'Davies-Bouldin (↓)': davies_bouldin_score(X_train_cluster, km_labels)
    })
    
    # --- Modelo Aglomerativo (Jerárquico) ---
    agg = AgglomerativeClustering(n_clusters=k)
    agg_labels = agg.fit_predict(X_train_cluster)
    
    resultados_clustering.append({
        'Modelo': 'Aglomerativo',
        'K': k,
        'Silhouette (↑)': silhouette_score(X_train_cluster, agg_labels),
        'Calinski-Harabasz (↑)': calinski_harabasz_score(X_train_cluster, agg_labels),
        'Davies-Bouldin (↓)': davies_bouldin_score(X_train_cluster, agg_labels)
    })

# Convertimos a DataFrame para ver la tabla comparativa
df_metricas_clusters = pd.DataFrame(resultados_clustering)
print("TABLA COMPARATIVA DE MÉTRICAS:")
print(df_metricas_clusters.sort_values(by=['Silhouette (↑)'], ascending=False).to_string(index=False))

# 4.4. SELECCIÓN DEL GANADOR
mejor_fila = df_metricas_clusters.loc[df_metricas_clusters['Silhouette (↑)'].idxmax()]
best_model_name = mejor_fila['Modelo']
best_k = int(mejor_fila['K'])

print(f"\n🥇 GANADOR DEL TORNEO: {best_model_name} con k={best_k}")

# Entrenamos el modelo ganador definitivo
if best_model_name == 'K-Means':
    best_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
else:
    best_model = AgglomerativeClustering(n_clusters=best_k)

final_labels_train = best_model.fit_predict(X_train_cluster)

# 4.5. INYECCIÓN SEGURA EN TRAIN Y TEST (Para los Modelo Predictivo)
print(" Inyectando etiquetas de clusters en el dataset...")

if hasattr(best_model, "predict"):
    final_labels_test = best_model.predict(X_test_cluster)
else:
    centroid_clf = NearestCentroid()
    centroid_clf.fit(X_train_cluster, final_labels_train)
    final_labels_test = centroid_clf.predict(X_test_cluster)

# One-Hot Encoding
train_dummies = pd.get_dummies(final_labels_train, prefix='Cluster_Group')
test_dummies = pd.get_dummies(final_labels_test, prefix='Cluster_Group')

# Alinear columnas por si el test no tiene algún cluster minoritario
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

# Unir a los datos originales
train_dummies.index = X_train.index
test_dummies.index = X_test.index
X_train = pd.concat([X_train, train_dummies], axis=1)
X_test = pd.concat([X_test, test_dummies], axis=1)

print(f"Fusión completada. Variables de cluster añadidas. Nuevas columnas: {list(train_dummies.columns)}")

--- Iniciando Optimización Avanzada de Clustering ---

🔍 Probando DBSCAN (Basado en Densidad)...
DBSCAN encontró 157 clusters y 904 puntos de ruido (outliers).
Motivo de descarte: En datos financieros continuos, DBSCAN suele agrupar casi todo en un solo clúster gigante o generar demasiado ruido. Pasamos a algoritmos particionales.

🏆 Iniciando Torneo: KMeans vs Jerárquico/Aglomerativo...

📊 TABLA COMPARATIVA DE MÉTRICAS:
      Modelo  K  Silhouette (↑)  Calinski-Harabasz (↑)  Davies-Bouldin (↓)
     K-Means  4        0.290332            2184.151725            1.299096
Aglomerativo  4        0.290332            2184.151725            1.299096
     K-Means  3        0.245855            1923.131621            1.749105
Aglomerativo  3        0.245855            1923.131621            1.749105
     K-Means  5        0.213722            1863.563072            1.590573
     K-Means  2        0.212887            1954.303375            1.957945
Aglomerativo  5        0.211763            1856.13

Cuando estudiamos Clustering en la teoría, se supone que lo mejor es donde los datos forman "islas" separadas (Silhouette > 0.70).

Pero en la vida real (y más en finanzas), los clientes no son islas;sino el el que cobra 1.500€ se mezcla con el que cobra 1.550€. El algoritmo ha cortado esa nube en 4 trozos. Como las fronteras entre los trozos se tocan y se solapan mucho, el Silhouette nos dice: "Oye, los grupos están muy pegados". Y es verdad, pero eso no significa que no sean útiles para tu modelo predictivo.

In [22]:
# 5. FUNCIÓN ENTRENAMIENTO (CON TIEMPOS Y GAP)
def entrenar_modelo(
        nombre_modelo,
        modelo,
        param_grid,
        X_train, X_test,
        y_train, y_test,
        usar_smote=False,
        usar_pca=False,
        threshold=None 
    ):
        
        steps = [("scaler", StandardScaler())]

        if usar_smote:
            steps.append(("smote", SMOTE(random_state=42)))
        if usar_pca:
            steps.append(("pca", PCA(n_components=0.95, random_state=42)))

        steps.append(("model", modelo))
        pipe = ImbPipeline(steps)

        param_grid_pipeline = {f"model__{k}": v for k,v in param_grid.items()}
        recall_scorer = make_scorer(recall_score, pos_label=1)

        # 1. MEDIR TIEMPO DE ENTRENAMIENTO 
        start_train = time.time()
        
        grid = GridSearchCV(pipe, param_grid_pipeline, cv=3, scoring=recall_scorer, n_jobs=-1)
        grid.fit(X_train, y_train)
        
        end_train = time.time()
        train_time = end_train - start_train  # Tiempo en segundos

        # 2. CÁLCULO DE THRESHOLD DINÁMICO (EN TRAIN PARA EVITAR LEAKAGE) 
        y_proba_train = grid.best_estimator_.predict_proba(X_train)[:, 1]
        
        if threshold is None:
            precisions, recalls, thresholds = precision_recall_curve(y_train, y_proba_train)
            f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
            best_idx = np.argmax(f1_scores)
            best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        else:
            best_threshold = threshold
            
        y_pred_train = (y_proba_train >= best_threshold).astype(int)

        # 3. PREDICCIÓN EN TEST Y TIEMPOS
        start_pred = time.time()
        y_proba_test = grid.best_estimator_.predict_proba(X_test)[:, 1]
        y_pred_test = (y_proba_test >= best_threshold).astype(int)
        end_pred = time.time()
        
        prediction_time = end_pred - start_pred

        # CÁLCULO DE METRICAS EN TRAIN vs TEST (Para ver Overfitting)
        train_acc = accuracy_score(y_train, y_pred_train)
        test_acc = accuracy_score(y_test, y_pred_test)
        gap = (train_acc - test_acc) * 100 

        # 4. OTRAS MÉTRICAS TEST
        roc = roc_auc_score(y_test, y_proba_test)
        recall1 = recall_score(y_test, y_pred_test, pos_label=1)
        precision1 = precision_score(y_test, y_pred_test, pos_label=1, zero_division=0)

        print("="*60)
        print(f"{nombre_modelo} | SMOTE={usar_smote} | PCA={usar_pca} | THRESH={best_threshold:.4f}")
        print(f" Tiempo Train: {train_time:.2f}s | Tiempo Pred: {prediction_time:.4f}s")
        print(f" Acc Train: {train_acc:.4f} | Acc Test: {test_acc:.4f} | GAP: {gap:.2f}%")
        print(" ROC-AUC:", round(roc,4))
        print(" Recall (Impago):", round(recall1,4))

        return {
            "Modelo": nombre_modelo,
            "SMOTE": usar_smote,
            "PCA": usar_pca,
            "Threshold": best_threshold,
            "Train_Time_Sec": train_time,     
            "Pred_Time_Sec": prediction_time,  
            "Train_Accuracy": train_acc,      
            "Test_Accuracy": test_acc,         
            "Overfitting_Gap_Pct": gap,         
            "ROC_AUC": roc,
            "Recall_1": recall1,
            "Precision_1": precision1
        }

In [ ]:
# 6. DEFINIR MODELOS
modelos = {
    "LogReg": (LogisticRegression(max_iter=1000, class_weight="balanced"), {"C":[0.01,0.1,1]}),
    "RandomForest": (RandomForestClassifier(random_state=42, class_weight="balanced"), {"n_estimators":[100,200]}),
    "DecisionTree": (DecisionTreeClassifier(random_state=42, class_weight="balanced"), {"max_depth":[None,5,10]}),
    "AdaBoost": (AdaBoostClassifier(random_state=42), {"n_estimators":[50,100]}),
    "XGBoost": (XGBClassifier(eval_metric="logloss", random_state=42, use_label_encoder=False),
                {"n_estimators":[100], "max_depth":[3,6]})
}

estimadores_base = [
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("dt", DecisionTreeClassifier(random_state=42)),
    ("nb", GaussianNB())
]

stacking = StackingClassifier(
    estimators=estimadores_base,
    final_estimator=LogisticRegression()
)

modelos["Stacking"] = (stacking, {"final_estimator__C":[0.1,1]})

# 7. EJECUCIÓN PARA TODAS LAS COMBINACIONES
combinaciones = [
    (False, False), # 1. Nada
    (True, False),  # 2. Solo SMOTE
    (False, True),  # 3. Solo PCA
    (True, True)    # 4. SMOTE + PCA
]

resultados_finales = []

for nombre, (modelo, grid) in modelos.items():
    for smote_flag, pca_flag in combinaciones:
        
        res = entrenar_modelo(
            nombre, modelo, grid,
            X_train, X_test,
            y_train, y_test,
            usar_smote=smote_flag,
            usar_pca=pca_flag,
            threshold=None
        )
        
        resultados_finales.append(res)

df_resultados = pd.DataFrame(resultados_finales)

In [14]:
#Resultados
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)       

print("\n=========== RESULTADOS FINALES ===========")
print(df_resultados.sort_values("Recall_1", ascending=False).to_string(index=False))


=========== RESULTADOS FINALES ===========
      Modelo  SMOTE   PCA  Threshold  Train_Time_Sec  Pred_Time_Sec  Train_Accuracy  Test_Accuracy  Overfitting_Gap_Pct  ROC_AUC  Recall_1  Precision_1
DecisionTree   True  True   0.492632        0.316428       0.001997        0.458928       0.452215             0.671305 0.576539  0.730769     0.139024
DecisionTree   True False   0.421367        0.123681       0.003000        0.487978       0.492494            -0.451605 0.589210  0.701923     0.144841
      LogReg   True  True   0.477728        0.135522       0.002999        0.536556       0.528012             0.854388 0.607111  0.647436     0.146271
    AdaBoost   True  True   0.493609        2.498908       0.009125        0.550836       0.547785             0.305139 0.600972  0.634615     0.150114
    AdaBoost  False  True   0.320126        1.427981       0.010013        0.592457       0.578909             1.354815 0.616547  0.596154     0.153719
    AdaBoost   True False   0.478397        

Los 5 Mejores Modelos (Equilibrio Recall / Estabilidad / AUC)
#------------------- DecisionTree | SMOTE=True | PCA=False

Recall: 0.7019 (Detecta al ~70% de los impagos).

ROC-AUC: 0.5892

GAP: -0.45% (¡Excelente! No hay nada de overfitting).

Por qué es el #1: Tiene el mejor Recall de los modelos estables. Un GAP negativo leve indica que generaliza de maravilla en datos nuevos.

#------------------- DecisionTree | SMOTE=True | PCA=True

Recall: 0.7307

ROC-AUC: 0.5765

GAP: 0.67%

Por qué es el #2: Técnicamente tiene un Recall un poquito más alto que el #1, pero su ROC-AUC es peor. Aun así, su GAP inferior al 1% lo hace extremadamente sólido.

#------------------- LogReg | SMOTE=True | PCA=True

Recall: 0.6474 (Caza al ~65% de los morosos).

ROC-AUC: 0.6071 (Muy bueno, superior a los árboles).

GAP: 0.85%

Por qué es el #3: Es el "Golden Standard". La Regresión Logística es explicable, rápida (0.26s), no tiene overfitting y mantiene un AUC por encima de 0.60, lo que gusta mucho en banca.

#------------------- AdaBoost | SMOTE=True | PCA=True

Recall: 0.6346

ROC-AUC: 0.6009

GAP: 0.30%

Por qué es el #4: Un modelo de ensamblado robusto que consigue un gran equilibrio. Mantiene el AUC por encima del 0.60 con un overfitting prácticamente nulo.

#------------------- AdaBoost | SMOTE=False | PCA=True

Recall: 0.5961

ROC-AUC: 0.6165 (¡El AUC más alto del Top 5!)

GAP: 1.35%

Por qué es el #5: Si el banco prefiere equivocarse un poco menos con los clientes buenos (mejor AUC) a costa de dejar escapar a algún moroso más (baja el Recall a casi el 60%), esta sería la elección.

Evaluamos la posibilidad de forzar un umbral conservador de 0.3 para maximizar la detección de morosos (Recall > 90%). Sin embargo, los resultados demostraron que esta política colapsaba la Exactitud global (Accuracy < 20%), provocando un rechazo masivo de clientes solventes. Por ello, optamos por el umbral dinámico basado en F1-Score (~0.48), que mantiene un equilibrio rentable para la entidad

Aunque a priori se esperaría un mejor rendimiento de los métodos de ensamblado (XGBoost, Random Forest), la naturaleza ruidosa de los datos crediticios provocó un severo overfitting en los modelos complejos. El Árbol de Decisión, al tener su profundidad limitada, actuó como un regularizador natural, capturando las reglas de negocio principales sin memorizar el ruido, logrando así el mejor equilibrio entre Recall y generalización.

**GRAFICOS**

Muestra en el eje Y cuánto acierta el modelo (Recall) y en el eje X cuánto "overfitting" tiene (el GAP).

Lo ideal es estar arriba a la izquierda (Mucho acierto, cero overfitting).

Verás a los modelos de Stacking y XGBoost perdidos por la derecha (mucho overfitting).

In [9]:
# GRÁFICO 1: RECALL vs OVERFITTING GAP
df_resultados['Config'] = df_resultados['Modelo'] + " | SMOTE:" + df_resultados['SMOTE'].astype(str) + " | PCA:" + df_resultados['PCA'].astype(str)

fig1 = px.scatter(
    df_resultados, 
    x="Overfitting_Gap_Pct", 
    y="Recall_1", 
    color="Modelo", 
    size="ROC_AUC", 
    hover_name="Config",
    hover_data={
        "Modelo": False,
        "Recall_1": ':.3f',
        "Overfitting_Gap_Pct": ':.2f',
        "ROC_AUC": ':.3f',
        "Threshold": ':.3f'
    },
    title="Capacidad de Detección (Recall) vs Estabilidad (Overfitting)",
    labels={
        "Overfitting_Gap_Pct": "Brecha Train-Test (% Overfitting) ➔ Peor",
        "Recall_1": "Tasa de Detección de Morosos (Recall) ➔ Mejor"
    },
    template="plotly_white"
)

# Añadimos líneas de referencia (Lo ideal es estar en el cuadrante superior izquierdo)
fig1.add_vline(x=5, line_width=2, line_dash="dash", line_color="red", annotation_text="Límite Peligro Overfitting")
fig1.add_hline(y=0.60, line_width=2, line_dash="dash", line_color="green", annotation_text="Objetivo Mínimo Recall")

fig1.show()

In [10]:
# GRÁFICO 2: RANKING TOP 5 MODELOS ROBUSTOS
# 1. Filtramos para quitar los que tienen mucho overfitting (tramposos)
df_robustos = df_resultados[df_resultados['Overfitting_Gap_Pct'] < 5.0].copy()

# 2. Ordenamos por Recall y nos quedamos con los 10 mejores
df_top10 = df_robustos.sort_values(by="Recall_1", ascending=True).tail(5) 

fig2 = go.Figure()

# Barra del Recall (Detección de impagos)
fig2.add_trace(go.Bar(
    y=df_top10['Config'],
    x=df_top10['Recall_1'],
    name='Recall (Detección Impagos)',
    orientation='h',
    marker=dict(color='rgba(50, 171, 96, 0.7)', line=dict(color='rgba(50, 171, 96, 1.0)', width=1))
))

# Barra del ROC-AUC (Calidad matemática general)
fig2.add_trace(go.Bar(
    y=df_top10['Config'],
    x=df_top10['ROC_AUC'],
    name='ROC-AUC (Calidad General)',
    orientation='h',
    marker=dict(color='rgba(128, 114, 255, 0.7)', line=dict(color='rgba(128, 114, 255, 1.0)', width=1))
))

fig2.update_layout(
    title='Top 5 Modelos Estables (Overfitting < 5%)',
    barmode='group',
    xaxis_title='Puntuación (0 - 1)',
    yaxis_title='Configuración del Modelo',
    template="plotly_white",
    legend=dict(x=0.8, y=0.1) # Movemos la leyenda abajo a la derecha
)

fig2.show()

In [ ]:
print("\n--- ENTRENANDO EL MODELO GANADOR DEFINITIVO ---")

# 1. Aislamos y entrenamos el mejor modelo (DecisionTree | SMOTE=True | PCA=False)
pasos_ganador = [
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("model", DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42))
]
modelo_ganador = ImbPipeline(pasos_ganador)

# Entrenamos
modelo_ganador.fit(X_train, y_train)

# Calculamos el umbral dinámico honesto (usando Train)
y_proba_train_ganador = modelo_ganador.predict_proba(X_train)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_train, y_proba_train_ganador)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_threshold_ganador = thresholds[np.argmax(f1_scores)]

# Predecimos en Test
y_proba_test_ganador = modelo_ganador.predict_proba(X_test)[:, 1]
y_pred_test_ganador = (y_proba_test_ganador >= best_threshold_ganador).astype(int)

print("Modelo ganador listo. Generando gráficos...")

# GRÁFICO 1: IMPORTANCIA DE LAS VARIABLES (FEATURE IMPORTANCE)
importancias = modelo_ganador.named_steps['model'].feature_importances_
nombres_variables = X_train.columns

df_importancias = pd.DataFrame({
    'Variable': nombres_variables,
    'Importancia': importancias
}).sort_values(by='Importancia', ascending=True).tail(10) # Cogemos el Top 10

fig1 = px.bar(
    df_importancias, 
    x='Importancia', 
    y='Variable', 
    orientation='h',
    title='Top 10 Variables más importantes para predecir el Impago',
    color='Importancia',
    color_continuous_scale='Reds'
)
fig1.update_layout(template="plotly_white", showlegend=False)
fig1.show()

# GRÁFICO 2: MATRIZ DE CONFUSIÓN INTERACTIVA

cm = confusion_matrix(y_test, y_pred_test_ganador)

fig2 = px.imshow(
    cm, 
    text_auto=True, 
    aspect="auto",
    labels=dict(x="Lo que dice el Modelo", y="La Realidad", color="Nº Clientes"),
    x=['Predice Pagador (0)', 'Predice Moroso (1)'],
    y=['Es Pagador (0)', 'Es Moroso (1)'],
    color_continuous_scale='Blues',
    title='Matriz de Confusión del Mejor Modelo'
)
fig2.update_xaxes(side="bottom")
fig2.update_layout(template="plotly_white")
fig2.show()

# GRÁFICO 3: CURVA ROC
fpr, tpr, _ = roc_curve(y_test, y_proba_test_ganador)
roc_auc = auc(fpr, tpr)

fig3 = px.area(
    x=fpr, y=tpr,
    title=f'Curva ROC (Área bajo la curva: {roc_auc:.3f})',
    labels=dict(x='Tasa de Falsos Positivos', y='Tasa de Verdaderos Positivos (Recall)'),
    width=700, height=500
)
fig3.add_shape(
    type='line', line=dict(dash='dash', color='red'),
    x0=0, x1=1, y0=0, y1=1
)
fig3.update_layout(template="plotly_white")
fig3.show()


--- ENTRENANDO EL MODELO GANADOR DEFINITIVO ---
✅ Modelo ganador listo. Generando gráficos...


(Por qué hay variables a cero)
Recuerda que obligamos al Árbol de Decisión a ser "bajito" (max_depth=5). Al limitarlo a un máximo de 5 niveles de profundidad, el árbol solo puede hacer unas pocas preguntas antes de tomar una decisión.
Como es muy "tacaño" con las preguntas que hace, solo usa las variables que separan a los morosos de forma radical en el primer corte. Si ya sabe que alguien tiene muchos créditos (Num_Creditos), igual ya no necesita preguntarle por su ratio de deuda (Ratio_Deuda_Ingresos) para catalogarlo como moroso. Las variables a cero no son "inútiles", simplemente el árbol encontró un atajo más rápido usando las de arriba.